Supervised Learning - Regression and hyperparameter tuning

In [22]:
# load blood-brain barrier dataset
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import numpy as np
df = pd.read_csv("BloodBrain.csv")
df.head()   


,tpsa,nbasic,negative,vsa_hyd,a_aro,weight,peoe_vsa.0,peoe_vsa.1,peoe_vsa.2,peoe_vsa.3,...,ctdh,ctaa,mchg,achg,rdta,n_sp2,n_sp3,o_sp2,o_sp3,logBBB
0,12.030000,1,0,167.06700,0,156.293,76.94749,43.44619,0.00000,0.000000,...,1,1,0.9241,0.9241,1.0000,0.000000,6.0255,0.000000,0.000000,1.08
1,49.330002,0,0,92.64243,6,151.165,38.24339,25.52006,0.00000,8.619013,...,2,2,1.2685,1.0420,1.0000,0.000000,6.5681,32.010201,33.613499,-0.40
2,50.529999,1,0,295.16700,15,366.485,58.05473,124.74020,21.65084,8.619013,...,1,4,1.2562,1.2562,0.2500,26.973301,10.8567,0.000000,27.545099,0.22
3,37.389999,0,0,319.11220,15,382.552,62.23933,124.74020,13.19232,21.785640,...,1,3,1.1962,1.1962,0.3333,21.706499,11.0017,0.000000,15.131600,0.14
4,37.389999,1,0,299.65800,12,326.464,74.80064,118.04060,33.00190,0.000000,...,1,3,1.2934,1.2934,0.3333,24.206100,10.8109,0.000000,15.133300,0.69


the BloodBrain dataset has 208 rows and 135 columns. 
208 compounds were analysed
for each compound, 134 molecular descriptors were calculated.
lobBBB is the log ratio of brain concentration to blood concentration of a compound: positive = higher concentration in liquor, crosses BBB well; 0 = equal concentration in liquor & blood; negative = higher concentration in blood, does not cross BBB well. 
The research question is to understand which kind of chemical/molecular properties influence "Liquorgängigkeit" (ability to cross blood brain barrier) of substances. 

In [23]:
# Split the dataset to create 75% training and 25% test data
from sklearn.model_selection import train_test_split
X = df.drop("logBBB", axis=1)
y = df["logBBB"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# select a learning method: random forest regression
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

# evaluate the model on the test data
y_test_pred = model.predict(X_test)
from sklearn.metrics import r2_score
r2_test = r2_score(y_test, y_test_pred)
print("Test R^2 Score:", r2_test)

# apply preprocessing (scaling/centering) is not necessary for random forest 
# since it splits data based on thresholds, 
# absolute size of values does not affect the model's performance. 

# Define a tuning grid for hyperparameters
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train) # fit only on the training data
print("Best Hyperparameters:", grid_search.best_params_)
best_model = grid_search.best_estimator_
y_test_pred_best = best_model.predict(X_test) # evaluate on the test data 
r2_test_best = r2_score(y_test, y_test_pred_best)
print("Test R^2 Score with Best Hyperparameters:", r2_test_best)    

Test R^2 Score: 0.39679091443404835
Fitting 5 folds for each of 27 candidates, totalling 135 fits
[CV] END max_depth=None, min_samples_split=2, n_estimators=100; total time=   0.7s
[CV] END max_depth=None, min_samples_split=2, n_estimators=100; total time=   0.7s
[CV] END max_depth=None, min_samples_split=2, n_estimators=100; total time=   0.7s
[CV] END max_depth=None, min_samples_split=2, n_estimators=100; total time=   0.7s
[CV] END max_depth=None, min_samples_split=2, n_estimators=100; total time=   0.8s
[CV] END max_depth=None, min_samples_split=2, n_estimators=200; total time=   1.7s
[CV] END max_depth=None, min_samples_split=2, n_estimators=200; total time=   1.7s
[CV] END max_depth=None, min_samples_split=2, n_estimators=200; total time=   1.7s
[CV] END max_depth=None, min_samples_split=5, n_estimators=100; total time=   0.6s
[CV] END max_depth=None, min_samples_split=2, n_estimators=200; total time=   1.7s
[CV] END max_depth=None, min_samples_split=2, n_estimators=200; total ti

R squared is 0.397 without hyperparameter tuning and 0.397 after tuning. So the tuning doesn't improve the model. (the model has already reached its performance ceiling on this dataset)

In [24]:
# perform 10-fold cross-validation 
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(model, X, y, cv=10, scoring='r2')
print("Cross-Validation R^2 Scores:", cv_scores)
print("Mean Cross-Validation R^2 Score:", cv_scores.mean())

Cross-Validation R^2 Scores: [ 0.6432249   0.54921367  0.50521257 -0.14337706 -0.04456488  0.3121887
  0.64728922 -0.13531521  0.34825303  0.51450548]
Mean Cross-Validation R^2 Score: 0.3196630421295152


R squared scores vary from -0.14 (very bad) to 0.65 (okay) across the ten folds. the model is sensitive to which samples end up in each fold! it is not generalizing well and probably overfitting.

In [25]:
# analyze performance values: print model performance metrics and feature importances
from sklearn.metrics import mean_squared_error, mean_absolute_error

y_pred = model.predict(X_test)

print("=== Model Performance Metrics ===")
print(f"R² Score:  {r2_score(y_test, y_pred):.4f}")
print(f"RMSE:      {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"MAE:       {mean_absolute_error(y_test, y_pred):.4f}")

# Print feature importances
print("\n=== Feature Importances ===")
importances = model.feature_importances_
feature_names = X.columns

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print(importance_df.to_string(index=False))

=== Model Performance Metrics ===
R² Score:  0.3968
RMSE:      0.5022
MAE:       0.3765

=== Feature Importances ===
             Feature  Importance
               fnsa3    0.194368
                tcnp    0.108463
most_positive_charge    0.043858
              tpsa.1    0.029097
            psa_npsa    0.028772
                 prx    0.023919
          slogp_vsa0    0.022837
               n_sp2    0.021094
          polar_area    0.020595
               clogp    0.020582
               pnsa3    0.019103
        peoe_vsa.1.1    0.017484
                rpcg    0.017071
               o_sp2    0.016736
           logp.o.w.    0.015767
               saaa3    0.012046
               mlogp    0.011604
               dpsa3    0.010732
                tpsa    0.009879
               fpsa3    0.009285
            smr_vsa0    0.009209
            smr_vsa3    0.008960
                rncg    0.008738
               n_sp3    0.008131
        peoe_vsa.3.1    0.007948
          andrewbind    0

In [26]:
# find out the range of y values to see if RMSD of 0.5 is a lot
print(y.describe())

count    208.000000
mean      -0.018894
std        0.779321
min       -2.150000
25%       -0.422500
50%        0.020000
75%        0.530000
max        1.640000
Name: logBBB, dtype: float64


RMSE is almost as large as the standard deviation of logBBB, so the model is only slightly better than guessing the mean for every compound. 